# Pipe Inspection Robot - Part 3: Crack detection with YOLO26n
#
# Train on this dataset: **Pipe Defects Detection** (Roboflow Universe,
# `gavril-birgauan`, CC BY 4.0) -> export to **NCNN at 320x320** -> run on the
# Raspberry Pi 4 CPU with `pi_detect.py`.
#
# What this notebook does, in order:
# 1. check the GPU  2. install  3. mount Drive  4. download the dataset
# 5. fix `data.yaml`  6. look at the data  7. settings  8. train
# 9. curves  10. test score  11. pictures  12. export NCNN  13. time it on CPU
# 14. download `best_ncnn_model.zip` + `best.pt`
#
# Run every cell top to bottom with **Shift+Enter**.

## 1. Check the GPU
Training needs a GPU. If `cuda available` is **False**, open
*Runtime -> Change runtime type -> T4 GPU -> Save* and start again from this cell.

In [ ]:
import os, sys, glob, platform, shutil, zipfile, time

print("python", sys.version.split()[0])
print("platform", platform.platform())
print("notebook working folder:", os.getcwd())

try:
    import torch
    print("torch", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
        print("gpu memory: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
    else:
        print("!! NO GPU - Runtime > Change runtime type > T4 GPU, then run all cells again")
except ImportError as exc:
    print("torch is not here yet - run section 2 first:", exc)

## 2. Install the tools (about 1 minute)
`ultralytics` trains the model, `roboflow` downloads the dataset, and
`pnnx` (pulled in with `ncnn`) converts the trained model to NCNN format.
If an install cell reports a CUDA/OOM problem later, just re-run this cell.

In [ ]:
!pip -q install --upgrade ultralytics roboflow ncnn pnnx opencv-python-headless pyyaml

import ultralytics
print("ultralytics", ultralytics.__version__)
print("pnnx found:", bool(shutil.which("pnnx")))

# which nano model can we actually get?  YOLO26n is the one we want; the
# fallbacks keep the notebook usable on an older ultralytics build.
from ultralytics import YOLO
for name in ("yolo26n.pt", "yolo11n.pt", "yolov8n.pt"):
    try:
        probe = YOLO(name)
        print("pretrained model available:", name)
        break
    except Exception as exc:
        print("  ", name, "not available:", str(exc)[:90])

## 3. Google Drive
Click **Allow** when asked. Everything worth keeping (`best.pt`,
`best_ncnn_model/`, the run folder with curves) is written to Drive, so a
Colab disconnect does not lose the training.

In [ ]:
TRAIN_DIR = "/content/drive/MyDrive/pipe_defects_yolo26n"
RUN_DIR = "/content/runs"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(TRAIN_DIR, exist_ok=True)
    print("drive ok ->", TRAIN_DIR)
except Exception as exc:
    print("!! Drive not available (%s). Files will only live in /content and be" % exc)
    print("   lost when this session ends. Mount Drive and re-run if that matters.")
    TRAIN_DIR = "/content/pipe_defects_yolo26n"
    os.makedirs(TRAIN_DIR, exist_ok=True)

## 4. Download the dataset
Get your key: **roboflow.com -> Settings -> API Keys -> Private API Key -> Copy**,
then paste it when the cell asks.

> The dataset is on **Roboflow Universe**, not in your own workspace yet.
> Open <https://universe.roboflow.com/gavril-birgauan/pipe-defects-detection>,
> click **Fork Dataset** to get a copy into your workspace, then put *your*
> workspace name below. Universe files also need **Download Dataset** (pick the
> *YOLOv8/YOLO11* format, not *Coco*).

Change `WORKSPACE`, `PROJECT` and `VERSION` to your own if you use your fork.

In [ ]:
import getpass

WORKSPACE = "YOUR-WORKSPACE"            # roboflow.com -> your project -> Overview
PROJECT   = "pipe-defects-detection"
VERSION   = 1
DATA_ROOT = "/content/pipe_defects"

api_key = os.environ.get("ROBOFLOW_API_KEY") or getpass.getpass("Roboflow API key: ")
if not api_key.strip():
    raise SystemExit("no key typed - copy it again from Settings > API Keys")

from roboflow import Roboflow

rf = Roboflow(api_key=api_key)
try:
    project = rf.workspace(WORKSPACE).project(PROJECT)
    dataset = project.version(VERSION).download("yolov8", location=DATA_ROOT)
except Exception as exc:
    print("download failed:", exc)
    print("checklist:")
    print("  1. the API key (Settings > API Keys > Private API Key)")
    print("  2. WORKSPACE is YOUR workspace name, not 'gavril-birgauan'")
    print("  3. you forked the Universe dataset first")
    print("  4. VERSION exists (check the Versions tab)")
    raise

print("downloaded to:", dataset.location)
for root, dirs, files in os.walk(DATA_ROOT):
    depth = root[len(DATA_ROOT):].count(os.sep)
    if depth <= 1:
        print("  ", root, "->", len(files), "files")

## 5. Fix `data.yaml`
Roboflow writes relative paths and sometimes leaves out the `test` split.
This cell points every split at a real absolute folder and, if there is no
`test` split, makes one (10% of the validation images) so section 10 can score
on unseen pictures.

In [ ]:
import yaml

# ---- find the yaml Roboflow wrote (the dataset sits in a nested folder)
candidates = sorted(glob.glob(os.path.join(DATA_ROOT, "**", "*.yaml"), recursive=True))
data_yaml_src = next((p for p in candidates if os.path.basename(p) == "data.yaml"), None) \
             or (candidates[0] if candidates else None)
if data_yaml_src is None:
    raise SystemExit("no yaml found under " + DATA_ROOT)

with open(data_yaml_src) as fh:
    data = yaml.safe_load(fh) or {}
print("raw data.yaml:", data)
YAML_DIR = os.path.dirname(data_yaml_src)

# ---- find the split folders by looking for them, not by trusting the paths.
# Roboflow writes train: ../train/images, which is relative to ultralytics'
# datasets_dir and therefore wrong as soon as we download somewhere else.
SPLIT_BY_FOLDER = {"train": "train", "valid": "val", "val": "val", "test": "test",
                   "training": "train", "validation": "val"}
resolved = {}
for images_dir in glob.glob(os.path.join(DATA_ROOT, "**", "images"), recursive=True):
    folder = os.path.basename(os.path.dirname(images_dir)).lower()
    key = SPLIT_BY_FOLDER.get(folder)
    sibling = os.path.join(os.path.dirname(images_dir), "labels")
    if key and os.path.isdir(sibling):
        resolved.setdefault(key, images_dir)

# ---- fall back to the paths in the yaml if the search came up short
# Roboflow writes the key as val: but some versions use valid:, so try both
for key, raw_keys in (("train", ["train"]), ("val", ["val", "valid"])):
    if key in resolved:
        continue
    raw = next((data.get(k) for k in raw_keys if data.get(k)), None)
    if not raw:
        continue
    path = raw if os.path.isabs(raw) else os.path.normpath(os.path.join(YAML_DIR, raw))
    for candidate in (os.path.join(path, "images"), path):
        if os.path.isdir(candidate):
            resolved.setdefault(key, candidate)
            break

# ---- make a test split if the download has none
if "test" not in resolved and "val" in resolved:
    val_images = sorted(p for p in glob.glob(os.path.join(resolved["val"], "*"))
                        if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")))
    move = val_images[:max(1, int(0.10 * len(val_images)))]
    if move:
        test_images = os.path.join(DATA_ROOT, "test", "images")
        test_labels = os.path.join(DATA_ROOT, "test", "labels")
        os.makedirs(test_images, exist_ok=True)
        os.makedirs(test_labels, exist_ok=True)
        for img in move:
            shutil.move(img, os.path.join(test_images, os.path.basename(img)))
            lbl = os.path.join(os.path.dirname(resolved["val"]), "labels",
                               os.path.splitext(os.path.basename(img))[0] + ".txt")
            if os.path.exists(lbl):
                shutil.move(lbl, os.path.join(test_labels, os.path.basename(lbl)))
        resolved["test"] = test_images
        print("built a test split from %d validation pictures" % len(move))

if "train" not in resolved:
    raise SystemExit("no train/images folder found under " + DATA_ROOT
                     + " - check the download above")
resolved.setdefault("val", resolved["train"])

data["train"] = resolved["train"]
data["val"] = resolved["val"]          # ultralytics wants the key 'val'
if "test" in resolved:
    data["test"] = resolved["test"]
data.pop("valid", None)

DATA_YAML = os.path.join(DATA_ROOT, "data_fixed.yaml")
with open(DATA_YAML, "w") as fh:
    yaml.safe_dump(data, fh, sort_keys=False)
print("splits:", {k: resolved[k] for k in ("train", "val") if k in resolved},
      "test:", resolved.get("test"))
print("classes:", data.get("nc"), data.get("names"))
print("wrote", DATA_YAML)

## 6. Look at the data
How many boxes of each class, and what the pictures actually look like.
The seven short names are the dataset author's abbreviations - look at the
pictures and note what each one means for your report:
`crack, hole, debris, buckling, jntoffs (joint offsets), obsc (obscured), utits (utility / damage)`.

In [ ]:
train_images = sorted(p for p in glob.glob(os.path.join(resolved["train"], "*"))
                      if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")))
print("training pictures:", len(train_images))

names = {int(k): v for k, v in data["names"].items()} if isinstance(data["names"], dict) \
        else {i: n for i, n in enumerate(data["names"])}
counts = {n: 0 for n in names.values()}
n_boxes = 0
for img in train_images:
    lbl = os.path.join(os.path.dirname(resolved["train"]), "labels",
                       os.path.splitext(os.path.basename(img))[0] + ".txt")
    if not os.path.exists(lbl):
        continue
    for line in open(lbl):
        if line.strip():
            counts[names.get(int(line.split()[0]), "?")] += 1
            n_boxes += 1

print("\nboxes per class in the training split (%d boxes total):" % n_boxes)
for n, c in sorted(counts.items(), key=lambda kv: -kv[1]):
    bar = "#" * int(40 * c / max(n_boxes, 1))
    print("  %-9s %6d  %s" % (n, c, bar))

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np

def show_labeled(paths, n=6, cols=3):
    if not paths:
        print("no pictures to show")
        return
    rows = (min(n, len(paths)) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = list(np.atleast_1d(axes).ravel())
    for ax, img_path in zip(axes, paths[:n]):
        lbl = os.path.join(os.path.dirname(resolved["train"]), "labels",
                           os.path.splitext(os.path.basename(img_path))[0] + ".txt")
        pil = Image.open(img_path).convert("RGB")
        ax.imshow(pil)
        if os.path.exists(lbl):
            w, h = pil.size
            for line in open(lbl):
                if not line.strip():
                    continue
                c, cx, cy, bw, bh = line.split()[:5]
                c, cx, cy, bw, bh = int(c), *[float(v) for v in (cx, cy, bw, bh)]
                x, y = (cx - bw / 2) * w, (cy - bh / 2) * h
                ax.add_patch(patches.Rectangle((x, y), bw * w, bh * h,
                                               fill=False, edgecolor="lime", lw=2))
                ax.text(x + 2, y + 12, names.get(c, "?"), color="lime", fontsize=10)
        ax.set_axis_off()
    for ax in axes[n:]:
        ax.set_axis_off()
    plt.tight_layout()
    plt.show()

show_labeled(train_images, n=6)

## 7. Settings
Everything you may want to change lives here.

| Setting | Value | Why |
|---|---|---|
| `MODEL` | `yolo26n.pt` | nano model - small enough to run on the Pi CPU |
| `IMGSZ` | 320 | about 4x less work than 640; keep it identical later on the Pi |
| `EPOCHS` | 100 | with `PATIENCE = 20` it stops early if it stops improving |
| `BATCH` | 64 | lower it to 32 or 16 if Colab runs out of memory |

**Fine-tuning shortcut:** to continue from a model you already trained,
upload `best.pt` to `/content/best.pt` and set `MODEL = "/content/best.pt"`
plus `EPOCHS = 50`.

In [ ]:
MODEL    = "yolo26n.pt"          # or "/content/best.pt" to fine-tune
IMGSZ    = 320
EPOCHS   = 100
BATCH    = 64                    # 32 or 16 if you hit out-of-memory
PATIENCE = 20
RUN_NAME = "yolo26n_320"

if not os.path.exists(MODEL):
    for fallback in ("yolo26n.pt", "yolo11n.pt", "yolov8n.pt"):
        try:
            YOLO(fallback)
            print("yolo26n.pt not here, using", fallback)
            MODEL = fallback
            break
        except Exception:
            continue
    else:
        raise SystemExit("no pretrained nano model could be downloaded - re-run section 2")

print("model:", MODEL, "| imgsz:", IMGSZ, "| epochs:", EPOCHS,
      "| batch:", BATCH, "| patience:", PATIENCE)

## 8. Train
About 20-40 minutes on a T4. Watch `mAP50` in the table grow; `loss` should
fall. The run folder is `$RUN_DIR/RUN_NAME`.

Leave the tab alone - closing the tab can kill the session.

In [ ]:
model = YOLO(MODEL)
model.train(
    data=DATA_YAML,
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    patience=PATIENCE,          # stop after 20 epochs with no improvement
    project=RUN_DIR,
    name=RUN_NAME,
    exist_ok=True,
    seed=0,
    workers=2,
    cache=False,                # True is faster if the dataset fits in RAM
    pretrained=True,
)

BEST = model.trainer.best
print("best weights:", BEST)

# a copy in Drive, so a disconnect cannot lose the training
os.makedirs(TRAIN_DIR, exist_ok=True)
shutil.copy(BEST, os.path.join(TRAIN_DIR, "best.pt"))
print("copied to", os.path.join(TRAIN_DIR, "best.pt"))

## 8b. Resume after a disconnect (only if you need it)
Re-run section 2 (install) and section 3 (Drive) first, then run this cell.
It restarts training from the last completed epoch in the Drive run folder.

In [ ]:
# uncomment and run only if the session dropped mid-training
# model = YOLO(MODEL)
# model.train(
#     data=DATA_YAML, imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, patience=PATIENCE,
#     project=RUN_DIR, name=RUN_NAME, exist_ok=True, resume=True,
# )
# BEST = model.trainer.best
# print("best weights:", BEST)

## 9. Curves and confusion matrix
What to look for:
* `box_loss` and `cls_loss` fall and flatten - if they still fall at the end,
  more epochs would help.
* `mAP50` climbs and plateaus. Over **0.5** is usable, **over 0.7** is good.
* precision low = too many false alarms (raise `--conf` later).
  recall low = missed defects (lower `--conf`, add more of your own pictures).
* large off-diagonal blocks in the confusion matrix = classes being mixed up.

In [ ]:
from IPython.display import Image as IPImage, display as ip_display
import pandas as pd

run_folder = os.path.join(RUN_DIR, RUN_NAME)
print("run folder:", run_folder, "| contents:", os.listdir(run_folder))

csv_path = os.path.join(run_folder, "results.csv")
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    cols = [c for c in df.columns if c.startswith("metrics/")]
    if cols:
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        axes[0].plot(df["train/box_loss"], label="train")
        if "val/box_loss" in df:
            axes[0].plot(df["val/box_loss"], label="val")
        axes[0].set_title("box loss"); axes[0].legend()
        axes[1].plot(df["train/cls_loss"], label="train")
        if "val/cls_loss" in df:
            axes[1].plot(df["val/cls_loss"], label="val")
        axes[1].set_title("class loss"); axes[1].legend()
        for c in ("metrics/mAP50(B)", "metrics/mAP50-95(B)"):
            if c in df:
                axes[2].plot(df[c], label=c.split("/")[-1])
        axes[2].set_title("mAP"); axes[2].legend()
        plt.tight_layout(); plt.show()
        print("best mAP50(B): %.3f" % df["metrics/mAP50(B)"].max()
              if "metrics/mAP50(B)" in df else "")

for picture in ("confusion_matrix.png", "confusion_matrix_normalized.png",
                "results.png", "PR_curve.png", "F1_curve.png"):
    path = os.path.join(run_folder, picture)
    if os.path.exists(path):
        print("\n", picture)
        ip_display(IPImage(filename=path))

## 10. Score on the test split
This is the honest number: the model has never seen these pictures.

In [ ]:
best = YOLO(BEST)

metrics = best.val(data=DATA_YAML, imgsz=IMGSZ, split="test", plots=True)
print("\n=== test results ===")
print("mAP50      : %.3f" % metrics.box.map50)
print("mAP50-95   : %.3f" % metrics.box.map)
print("precision  : %.3f" % metrics.box.mp)
print("recall     : %.3f" % metrics.box.mr)

print("\nper class (name, mAP50, mAP50-95, precision, recall):")
# ap50/p/r are positional in ap_class_index order, maps is id-indexed, so walk
# the classes that are actually present and index ap50/p/r by position
print("  %-9s %6s %8s %9s %7s" % ("class", "mAP50", "mAP50-95", "precision", "recall"))
for pos, cid in enumerate(metrics.box.ap_class_index):
    name = metrics.names[int(cid)]
    def at(attr):
        try:
            return "%.3f" % float(getattr(metrics.box, attr)[pos])
        except Exception:
            return "-"
    print("  %-9s %6s %8s %9s %7s"
          % (name, at("ap50"), "%.3f" % float(metrics.box.maps[int(cid)]),
             at("p"), at("r")))

## 11. The model's own pictures
Six test images with the boxes the model drew. Wrong boxes here are normal -
this is what you are looking for in the pipes later on.

In [ ]:
test_images = sorted(p for p in glob.glob(os.path.join(resolved.get("test", resolved["val"]), "*"))
                     if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")))

from IPython.display import display as ip_display2
for img_path in test_images[:6]:
    result = best.predict(img_path, imgsz=IMGSZ, conf=0.40, verbose=False)[0]
    ip_display2(result.plot())
    print(os.path.basename(img_path), "->",
          [names[int(b.cls)] for b in result.boxes] or "nothing found")

## 12. Export to NCNN at 320x320
NCNN is the format built for ARM CPUs, which is what makes this usable on the
Pi 4. The export returns the folder it created - it contains
`model.ncnn.param` and `model.ncnn.bin`, and it sits next to `best.pt`.

In [ ]:
# export() returns the path of the folder it created
NCNN_DIR = best.export(format="ncnn", imgsz=IMGSZ)
if NCNN_DIR is None:
    NCNN_DIR = "/content/best_ncnn_model"
elif os.path.isfile(NCNN_DIR):          # some versions return the .pt file
    NCNN_DIR = os.path.join(os.path.dirname(NCNN_DIR), "best_ncnn_model")

print("ncnn folder:", NCNN_DIR)
for f in sorted(os.listdir(NCNN_DIR)):
    print("  ", f, "%.1f kB" % (os.path.getsize(os.path.join(NCNN_DIR, f)) / 1024))
print("\nclasses kept by the export:", YOLO(NCNN_DIR, task="detect").names)

## 13. Does the NCNN model actually run on a CPU?
Colab's CPU is roughly in the same ballpark as a Pi 4 core. If this is 30 ms
or less you should see 4-8 FPS on the Pi (camera capture and streaming take
the rest).

In [ ]:
import cv2
ncnn_model = YOLO(NCNN_DIR, task="detect")

test_pic = test_images[0] if test_images else train_images[0]
frame = cv2.imread(test_pic)
ncnn_model.predict(frame, imgsz=IMGSZ, verbose=False)      # warm-up

N = 20
t0 = time.time()
for _ in range(N):
    ncnn_model.predict(frame, imgsz=IMGSZ, verbose=False)
ms = (time.time() - t0) / N * 1000

print("image size     :", frame.shape)
print("inference (cpu): %.1f ms per image  ->  %.1f FPS ceiling"
      % (ms, 1000 / max(ms, 1e-6)))
print("\nif this is much over 100 ms, on the Pi use the NCNN folder (not best.pt),")
print("keep --imgsz 320, and close everything else that is running.")

## 14. Download for the Pi
You need:
* **`best_ncnn_model.zip`** -> the Pi runs this
* **`best.pt`** -> keep it for fine-tuning (and for the laptop)

Copy the zip to the Pi:

```
scp best_ncnn_model.zip jellyfish@JellyFish.local:~/pipe_robot/pi/
```

On the Pi:

```
cd ~/pipe_robot/pi
unzip best_ncnn_model.zip -d best_ncnn_model
source ~/robot-env/bin/activate
python3 pi_detect.py --model best_ncnn_model --src pipe_run1.mp4
```

`unzip -d best_ncnn_model` matters: the zip holds `model.ncnn.param` and
`model.ncnn.bin`, and the `-d` puts them in the folder name that pi_detect.py
expects.

In [ ]:
# Zip only what the Pi needs.  The export folder also holds model.pt and the
# pnnx intermediates, and ultralytics picks the model with a *.param glob, so
# shipping the lot would risk it loading the wrong file.
WANTED = ["model.ncnn.param", "model.ncnn.bin", "metadata.yaml"]
STAGE = "/content/ncnn_package/best_ncnn_model"
shutil.rmtree("/content/ncnn_package", ignore_errors=True)
os.makedirs(STAGE, exist_ok=True)
missing = [f for f in WANTED if not os.path.exists(os.path.join(NCNN_DIR, f))]
if missing:
    print("!! missing from the export folder:", missing)
    print("   the folder holds:", os.listdir(NCNN_DIR))
for name in WANTED:
    src_file = os.path.join(NCNN_DIR, name)
    if os.path.exists(src_file):
        shutil.copy(src_file, os.path.join(STAGE, name))
        print("   packed", name)

NCNN_ZIP = "/content/best_ncnn_model.zip"
if os.path.exists(NCNN_ZIP):
    os.remove(NCNN_ZIP)
# the zip holds the three files, so `unzip best_ncnn_model.zip -d best_ncnn_model`
# on the Pi produces exactly the layout pi_detect.py looks for
with zipfile.ZipFile(NCNN_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in WANTED:
        src_file = os.path.join(STAGE, name)
        if os.path.exists(src_file):
            zf.write(src_file, name)

shutil.copy(BEST, "/content/best.pt")
print("\nbest_ncnn_model.zip: %.1f MB" % (os.path.getsize(NCNN_ZIP) / 1e6))
print("best.pt            : %.1f MB" % (os.path.getsize('/content/best.pt') / 1e6))
with zipfile.ZipFile(NCNN_ZIP) as zf:
    print("zip contains:", zf.namelist())

# keep everything in Drive too
shutil.copy(NCNN_ZIP, TRAIN_DIR)
shutil.copy(BEST, os.path.join(TRAIN_DIR, "best.pt"))
run_copy = os.path.join(TRAIN_DIR, RUN_NAME)
if os.path.isdir(run_folder) and not os.path.isdir(run_copy):
    shutil.copytree(run_folder, run_copy)
print("saved to Drive:", TRAIN_DIR)

try:
    from google.colab import files
    files.download(NCNN_ZIP)
    files.download("/content/best.pt")
except Exception:
    print("\nnot a Colab runtime - collect the files from", NCNN_ZIP, "and /content/best.pt")